# Data Wrangling - Hematin Finance Dataset
## Pengertian Data Wrangling
Data wrangling adalah proses mengubah data mentah menjadi data yang lebih bersih, rapi, konsisten, dan siap digunakan untuk analisis.

Tahapan utama data wrangling terdiri dari:

1. Gathering → Mengumpulkan data
2. Assessing → Mengecek kondisi data
3. Cleaning → Membersihkan dan memperbaiki data


Kami menggunakan 2 dataset (budgetwise_finance_dataset dan Daily Household Transactions) yang kami temukan dari Kaggle. Kemudian kami gabungkan 2 dataset tersebut agar data yang diperoleh lebih lengkap.  

| Dataset | Sumber | Keterangan |
|---------|--------|------------|
| `budgetwise_finance_dataset.csv` | Kaggle | Data transaksi keuangan personal (internasional) |
| `Daily Household Transactions.csv` | Kaggle | Data transaksi rumah tangga harian (asal India, dalam INR) |


# Wrangling Dataset 1 (budgetwise_finance_dataset)

# 1. GATHERING (Pengumpulan Data)

Tahap gathering dilakukan ketika dataset dibaca menggunakan pandas.

In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv(
    "/content/sample_data/budgetwise_finance_dataset.csv"
)

## Penjelasan:  
- pd.read_csv() membaca file CSV dan menyimpannya ke dalam DataFrame pandas.
Library yang diimpor:

- pandas → manipulasi data tabular
numpy → operasi numerik (NaN, array, dsb.)
re → regular expression untuk pencocokan pola teks

# 2. ASSESSING (Pengecekan Data)

Tahap assessing digunakan untuk memeriksa kondisi awal dataset sebelum dibersihkan.


## 2.1 Melihat Data Awal

In [2]:
print("=" * 60)
print("DATA AWAL")
print("=" * 60)

print("\n5 Data Pertama:")
print(df.head())


DATA AWAL

5 Data Pertama:
  transaction_id user_id        date transaction_type       category amount  \
0          T4999    U018  2023-04-25          Expense       Educaton   3888   
1         T12828    U133  08/05/2022          Expense           rent    649   
2          T7403    U091    31-12-23           Income      Freelance  13239   
3         T12350    U097         NaN          Expense            Fod   6299   
4          T7495    U088  10/28/2022          Expense  entertainment   2287   

    payment_mode   location             notes  
0           card  Ahmedabad     Movie tickets  
1            NaN  Hyderabad            asdfgh  
2            Csh        BAN             Books  
3  Bank Transfer  AHMEDABAD  Electricity bill  
4           CARD  Hyderabad               NaN  


## Menampilkan 5 data pertama dari dataset.
- Kolom berisi: tanggal, kategori, metode pembayaran, lokasi, nominal, catatan
- Terdapat typo (Educaton, fod) yang perlu diperbaiki
- Terdapat inkonsistensi penulisan (Debitcard, Healthcare vs health)

## 2.2 Melihat Informasi Dataset

In [3]:
print("\nInfo Dataset:")
print(df.info())


Info Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15900 entries, 0 to 15899
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   transaction_id    15900 non-null  object
 1   user_id           15900 non-null  object
 2   date              15414 non-null  object
 3   transaction_type  15900 non-null  object
 4   category          15615 non-null  object
 5   amount            15609 non-null  object
 6   payment_mode      15092 non-null  object
 7   location          14638 non-null  object
 8   notes             13079 non-null  object
dtypes: object(9)
memory usage: 1.1+ MB
None


Output dari perintah ini memberikan gambaran teknis tentang dataset. Kita bisa melihat jumlah total baris, jumlah kolom, tipe data tiap kolom, dan berapa banyak data yang tidak kosong di setiap kolom. Kolom amount bertipe float64 (angka desimal), sedangkan kolom lain bertipe object (teks). Jika jumlah non-null lebih kecil dari total baris, berarti ada missing value di kolom tersebut.

## 2.3 Mengecek Missing Value

In [4]:
print("\nMissing Value:")
print(df.isnull().sum())


Missing Value:
transaction_id         0
user_id                0
date                 486
transaction_type       0
category             285
amount               291
payment_mode         808
location            1262
notes               2821
dtype: int64


 Hasilnya menunjukkan bahwa kolom notes paling banyak kosong karena memang bersifat opsional, sementara kolom amount dan date juga ada yang kosong meski jumlahnya lebih sedikit — ini yang perlu ditangani dengan hati-hati karena keduanya sangat penting untuk analisis.

# 3. CLEANING (Pembersihan Data)
Tahap cleaning dilakukan untuk membersihkan dan memperbaiki dataset.

# 3.1 Merapikan Nama Kolom


In [5]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

print("\nNama Kolom:")
print(df.columns)



Nama Kolom:
Index(['transaction_id', 'user_id', 'date', 'transaction_type', 'category',
       'amount', 'payment_mode', 'location', 'notes'],
      dtype='object')


Nama kolom dibersihkan agar lebih mudah dipanggil dalam kode. Spasi di awal dan akhir nama kolom dihapus dengan .strip(), semua huruf diubah ke huruf kecil dengan .lower(), dan spasi antar kata diganti dengan underscore menggunakan .str.replace(' ', '_'). Hasilnya, nama seperti "Payment Mode" berubah menjadi "payment_mode". Ini penting agar pemanggilan kolom seperti df['payment_mode'] tidak menyebabkan error.

# 3.2 Membersihkan Text


In [6]:
for col in df.select_dtypes(include='object').columns:

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
    )

Seluruh kolom bertipe teks diiterasi satu per satu, lalu setiap nilai dibersihkan dari spasi tersembunyi di awal dan akhir. Ini menangani kasus seperti nilai "  Cash  " yang seharusnya hanya "Cash", karena spasi ekstra bisa menyebabkan pencocokan data gagal.

# 3.3 Menangani Missing Value  
Mengubah String Kosong Menjadi NaN

In [7]:
df = df.replace(r'^\s*$', np.nan, regex=True)


In [8]:
df['category'] = df['category'].fillna('unknown')
df['payment_mode'] = df['payment_mode'].fillna('unknown')
df['location'] = df['location'].fillna('unknown')
df['notes'] = df['notes'].fillna('no notes')

In [9]:
# hapus data penting kosong
df = df.dropna(subset=['amount', 'date'])


String kosong (yang mungkin lolos dari pengecekan isnull()) diubah menjadi NaN menggunakan regex. Setelah itu, setiap kolom ditangani sesuai tingkat kepentingannya. Kolom category, payment_mode, dan location diisi dengan nilai 'unknown' karena data masih bisa diproses meski informasinya tidak lengkap. Kolom notes diisi 'no notes' karena memang bersifat opsional. Sebaliknya, baris yang tidak memiliki nilai amount atau date langsung dihapus, karena kedua kolom ini wajib ada untuk analisis transaksi keuangan.

# 3.4 Membersihkan Kategori

In [10]:
df['category'] = (
    df['category']
    .astype(str)
    .str.lower()
    .str.strip()
)

# typo & variasi kategori
df['category'] = df['category'].replace({

    'fod': 'food',
    'foods': 'food',

    'educaton': 'education',

    'entertainment ': 'entertainment',
    'movie': 'entertainment',

    'healthcare': 'health',
    'medical': 'health',

    'rent ': 'rent',
    'bills': 'utilities'
})


Semua nilai kategori diubah ke huruf kecil terlebih dahulu agar pencocokan tidak sensitif terhadap huruf besar/kecil. Kemudian typo dan variasi penulisan diperbaiki secara manual menggunakan .replace(). Misalnya, 'fod' dan 'foods' keduanya diperbaiki menjadi 'food', 'healthcare' dan 'medical' disatukan menjadi 'health', dan 'movie' dikelompokkan ke dalam 'entertainment' karena keduanya merujuk pada hiburan.

# 3.5 Standardisasi Kategori Final

In [11]:
kategori_mapping = {

    # makanan
    'food': 'Makanan/Minuman',

    # kesehatan
    'health': 'Kesehatan',

    # hiburan
    'entertainment': 'Hiburan',
     'travel': 'Hiburan',

    # utilitas
    'utilities': 'Tagihan/Utilitas',
    'rent': 'Tagihan/Utilitas',

    # lainnya
    'education': 'Pendidikan',
    'transport': 'Transportasi',
    'shopping': 'Belanja',
    'unknown': 'Lain-lain'
}

df['kategori'] = df['category'].map(kategori_mapping)
df['kategori'] = df['kategori'].fillna('Lain-lain')

Kategori dalam bahasa Inggris dipetakan ke label yang lebih mudah dipahami dalam Bahasa Indonesia. rent dan utilities digabungkan ke dalam satu kategori 'Tagihan/Utilitas' karena keduanya adalah pengeluaran rutin yang sifatnya sama. Kategori yang tidak cocok dengan mapping manapun secara otomatis diisi 'Lain-lain'.

# 3.6 Membersihkan Payment Mode

In [12]:
df['payment_mode'] = (
    df['payment_mode']
    .astype(str)
    .str.lower()
    .str.strip()
)

df['payment_mode'] = df['payment_mode'].replace({

    'csh': 'cash',
    'card ': 'card',
    'debitcard': 'debit card',
    'creditcard': 'credit card'
})

Typo dan penulisan yang tidak konsisten pada metode pembayaran diperbaiki. 'csh' diperbaiki menjadi 'cash', dan 'debitcard' dipisah menjadi 'debit card' agar formatnya konsisten dengan 'credit card'

# 3.7 Format Tanggal

In [13]:
df['date'] = pd.to_datetime(
    df['date'],
    format='mixed',
    errors='coerce'
)

# hapus tanggal invalid
df = df.dropna(subset=['date'])


Kolom tanggal yang masih berupa teks diubah menjadi tipe data datetime menggunakan pd.to_datetime(). Parameter format='mixed' memungkinkan pandas mengenali berbagai format tanggal secara otomatis, seperti "2023-01-01", "01/01/2023", atau "Jan 1, 2023". Jika ada format yang tidak dikenali, nilainya diubah menjadi NaT (Not a Time) oleh errors='coerce', lalu baris tersebut dihapus. Kolom yang sudah bertipe datetime memudahkan analisis berbasis waktu seperti tren bulanan atau tahunan.

# 3.8 Membersihkan Amount

In [14]:
df['amount'] = pd.to_numeric(
    df['amount'],
    errors='coerce'
)
df = df.dropna(subset=['amount'])
df = df[df['amount'] >= 0]

Kolom amount dikonversi ke tipe numerik. Jika ada nilai yang tidak bisa dikonversi (misalnya teks acak), nilainya menjadi NaN dan baris tersebut dihapus. Baris dengan amount negatif juga dihapus karena transaksi pengeluaran tidak mungkin bernilai minus.

# 3.9 Menghapus Duplicate

In [15]:
df = df.drop_duplicates()

Baris yang sepenuhnya identik dihapus agar tidak terjadi perhitungan ganda saat analisis. Misalnya, jika satu transaksi terekam dua kali, hasilnya akan seolah-olah total pengeluaran lebih besar dari yang sebenarnya.

# 3.10 Mengubah Nama Kolom Final

In [16]:
df = df.rename(columns={

    'date': 'tanggal_transaksi',
    'amount': 'total_pengeluaran',
    'notes': 'nama_produk'
})

df_final = df[[
    'tanggal_transaksi',
    'nama_produk',
    'kategori',
    'total_pengeluaran'
]]



Nama kolom diubah ke Bahasa Indonesia agar lebih mudah dipahami, dan hanya empat kolom yang diperlukan yang diambil untuk dataset final.

## 3.11 Menghapus Outlier

In [17]:
# menghapus outlier
df_final = df_final[
    df_final['total_pengeluaran'] < 10000000
]

Pada tahap cleaning data dilakukan pengecekan outlier pada kolom `total_pengeluaran`. Outlier merupakan data yang memiliki nilai sangat jauh dari mayoritas data lainnya sehingga dapat memengaruhi hasil analisis dan visualisasi.

Pada dataset ditemukan beberapa nilai yang tidak wajar seperti `999999999`, yang diduga merupakan error input atau dummy value. Nilai tersebut dapat menyebabkan grafik dan insight menjadi tidak akurat.

Oleh karena itu dilakukan proses pembersihan outlier menggunakan metode filtering agar hanya data dengan nilai yang masih realistis yang digunakan dalam analisis.


# 3.12 Cleaning Nama Produk

In [18]:
df_final['nama_produk'] = (
    df_final['nama_produk']
    .astype(str)
    .str.lower()
    .str.strip()
)


In [19]:
df_final['nama_produk'] = df_final['nama_produk'].replace({

    'nan': np.nan,
    'none': np.nan,
    'null': np.nan,
    '...': np.nan,
    '.': np.nan,
    '-': np.nan,
    'unknown': np.nan,
    'no notes': np.nan
})

Teks yang tidak bermakna seperti 'nan', 'null', '-', atau '...' diubah menjadi NaN agar tidak dianggap sebagai nama produk yang valid.

# 3.13 Deteksi Garbage / Noise Text

In [20]:
garbage_keywords = [
    'xyz123',
    'misc',
    'random',
    'unknown',
    'test',
    'dummy',
    'sample',
    'abcd',
    'xxx',
    'null',
    'na',
    'n/a'
]

df_final.loc[
    df_final['nama_produk'].isin(garbage_keywords),
    'nama_produk'
] = np.nan

In [21]:
anomali_patterns = [
    r'^[asdfghjkl]+$',
    r'^[qwertyuiop]+$',
    r'^[zxcvbnm]+$'
]

for pattern in anomali_patterns:

    df_final.loc[
        df_final['nama_produk'].str.match(
            pattern,
            na=False
        ),
        'nama_produk'
    ] = np.nan

Kode diatas mendeteksi dua jenis data tidak valid. Pertama adalah garbage keywords, yaitu kata-kata placeholder yang biasa dimasukkan sembarangan saat mengisi form seperti 'test', 'dummy', atau 'sample'. Kedua adalah keyboard random, yaitu pola penekanan keyboard asal-asalan seperti 'asdfgh' atau 'qwerty' yang jelas bukan nama produk nyata. Semua nilai yang cocok dengan pola ini diubah menjadi NaN.
Selain itu, teks yang terlalu pendek (tiga karakter atau kurang) juga dihapus karena tidak memberikan informasi yang bermakna.

# 3.14 Menghapus Teks Pendek

In [22]:
df_final['nama_produk'] = df_final['nama_produk'].apply(
    lambda x: np.nan
    if isinstance(x, str) and len(x.strip()) <= 3
    else x
)

df_final['nama_produk'] = df_final['nama_produk'].apply(
    lambda x: np.nan
    if isinstance(x, str)
    and any(char.isdigit() for char in x)
    and len(x) <= 8
    else x
)

df_final['nama_produk'] = df_final['nama_produk'].replace(
    r'^\s*$',
    np.nan,
    regex=True
)

Pada dataset, terdapat teks yang terlalu pendek sehingga lebih baik disisihkan saja dan menghapus string yang kosong.

# 3.15 Rule-Based Category Correction

In [23]:
import re

def perbaiki_kategori(row):

    nama = str(row['nama_produk']).lower()


    hiburan = [
        'netflix',
        'spotify',
        'youtube',
        'disney',
        'movie',
        'film',
        'game',
        'travel',
        'vacation',
        'holiday'
    ]

    kesehatan = [
        'doctor',
        'dokter',
        'hospital',
        'clinic',
        'medicine',
        'medical',
        'medicines',
        'vitamin',
        'obat',
        'pharmacy',
        'apotek',
        'gym',
        'fitness',
        'workout'
    ]

    transport = [
        'train',
        'ticket',
        'uber',
        'grab',
        'gojek',
        'gocar',
        'taxi',
        'bus',
        'petrol',
        'fuel',
        'diesel',
        'transport'
    ]

    utilitas = [
        'wifi',
        'router',
        'internet',
        'electricity',
        'rent',
        'bill',
        'subscription',
        'emi',
        'loan'
    ]

    makanan = [
        'food',
        'coffee',
        'pizza',
        'burger',
        'restaurant',
        'cafe',
        'lunch',
        'breakfast',
        'dinner',
        'meal',
        'snack',
        'tea',
        'milk',
        'rice',
        'chicken'
    ]

    belanja = [
        'shopping',
        'purchase',
        'buy',
        'grocery',
        'groceries',
        'market'
    ]

    pendidikan = [
        'book',
        'books',
        'study',
        'school',
        'college',
        'course',
        'online course',
        'udemy',
        'coursera',
        'bootcamp',
        'class',
        'education'
    ]


    def contains_keyword(keywords):

        return any(
            re.search(
                rf'\b{re.escape(word)}\b',
                nama
            )
            for word in keywords
        )


    # ATM / CASH
    if contains_keyword([
        'atm',
        'withdrawal',
        'cash'
    ]):
        return 'Keuangan'

    # INVESTASI
    elif contains_keyword([
        'fixed deposit',
        'investment',
        'invest'
    ]):
        return 'Investasi'

    # PENDIDIKAN
    elif contains_keyword(pendidikan):
        return 'Pendidikan'

    # SHOPPING
    elif contains_keyword(belanja):
        return 'Belanja'

    # HIBURAN
    elif contains_keyword(hiburan):
        return 'Hiburan'

    # KESEHATAN
    elif contains_keyword(kesehatan):
        return 'Kesehatan'

    # TRANSPORTASI
    elif contains_keyword(transport):
        return 'Transportasi'

    # UTILITAS
    elif contains_keyword(utilitas):
        return 'Tagihan/Utilitas'

    # MAKANAN
    elif contains_keyword(makanan):
        return 'Makanan/Minuman'

    # DEFAULT
    return row['kategori']


df_final['kategori'] = df_final.apply(
    perbaiki_kategori,
    axis=1
)

Fungsi ini bekerja dengan memeriksa nama produk dari setiap baris lalu menentukan apakah kategori yang sudah ada perlu dikoreksi. Caranya adalah dengan memeriksa apakah nama produk mengandung kata kunci tertentu. Fungsi pembantu contains_keyword() menggunakan re.search() dengan pola \b...\b yang berarti "kata utuh" — ini mencegah kesalahan pencocokan, misalnya kata 'tea' tidak akan cocok dengan 'teacher'. Kode memeriksa secara berurutan dari atas ke bawah, dan prioritas tertinggi diberikan kepada aturan "override" seperti ATM/cash, buku, dan belanja yang langsung diarahkan ke 'Lain-lain' meskipun mungkin cocok dengan pola lain. Setelah semua kondisi dicek, jika tidak ada yang cocok, kategori dari baris tersebut dibiarkan seperti semula (return row['kategori']). Fungsi dijalankan ke seluruh baris dengan df_final.apply(perbaiki_kategori, axis=1) — axis=1 berarti fungsi dijalankan per baris, bukan per kolom.

# 3.16 Menghapus Missing Value Final

In [24]:
df_final = df_final.dropna(
    subset=[
        'tanggal_transaksi',
        'nama_produk',
        'kategori',
        'total_pengeluaran'
    ]
)

# hapus duplicate
df_final = df_final.drop_duplicates()

Setelah seluruh proses cleaning nama produk yang panjang, sangat mungkin ada baris-baris baru yang kini memiliki NaN di kolom nama_produk karena nilainya dibersihkan. Kode ini memastikan semua baris yang memiliki missing value di salah satu dari empat kolom utama dihapus. Setelah itu duplikat yang mungkin muncul akibat proses cleaning juga dihapus kembali.

# 3.17 Dataset yang dihasilkan

In [25]:
print("\n" + "=" * 60)
print("HASIL DATA WRANGLING 1")
print("=" * 60)

print("\n5 Data Final 1:")
print(df_final.head())

print("\nInfo Dataset Final 1:")
print(df_final.info())

print("\nMissing Value Final 1:")
print(df_final.isnull().sum())

print("\nJumlah Duplicate 1:")
print(df_final.duplicated().sum())

print("\nDistribusi Kategori 1:")
print(df_final['kategori'].value_counts())


HASIL DATA WRANGLING 1

5 Data Final 1:
   tanggal_transaksi        nama_produk          kategori  total_pengeluaran
0         2023-04-25      movie tickets           Hiburan             3888.0
2         2023-12-31              books        Pendidikan            13239.0
9         2023-11-10     gym membership         Kesehatan            59543.0
13        2022-02-24   electricity bill  Tagihan/Utilitas            23760.0
14        2023-08-28  restaurant dinner   Makanan/Minuman              898.0

Info Dataset Final 1:
<class 'pandas.core.frame.DataFrame'>
Index: 7134 entries, 0 to 15897
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   tanggal_transaksi  7134 non-null   datetime64[ns]
 1   nama_produk        7134 non-null   object        
 2   kategori           7134 non-null   object        
 3   total_pengeluaran  7134 non-null   float64       
dtypes: datetime64[ns](1), float64(1), obj

# Penjelasan Hasil Data Wrangling

Bagian ini menampilkan 5 data pertama setelah cleaning selesai.

Hasil yang terlihat:
nama kolom sudah rapi
kategori sudah distandardisasi
tanggal sudah menjadi format datetime
amount sudah numerik
data noise sudah hilang

Bagian ini merupakan hasil akhir setelah proses data wrangling selesai dilakukan.
Tujuannya adalah memastikan dataset sudah:
- bersih
- konsisten
- tidak memiliki missing value
- tidak memiliki duplicate
- siap dianalisis



# 3.17 Menyimpan Dataset Bersih

In [26]:
output_file = "cleaned_budgetwise_finance_dataset.csv"

df_final.to_csv(
    output_file,
    index=False
)

print("\nData wrangling 1 selesai!")
print(f"Dataset 1 berhasil disimpan: {output_file}")


Data wrangling 1 selesai!
Dataset 1 berhasil disimpan: cleaned_budgetwise_finance_dataset.csv


Dataset sudah tersimpan dan siap untuk digabungkan.

# Wrangling Dataset 2 (Daily Household Transactions)

# 1. Gathering

## 1.1 Load Dataset

In [27]:
df1 = pd.read_csv('/content/sample_data/Daily Household Transactions.csv')



# 2. Assessing

## 2.1 Cek Data Awal

In [28]:
print("=" * 60)
print("DATA AWAL")
print("=" * 60)

print("\n5 Data Pertama:")
print(df1.head())

print("\nInfo Dataset:")
print(df1.info())

DATA AWAL

5 Data Pertama:
                  Date                   Mode        Category  \
0  20/09/2018 12:04:08                   Cash  Transportation   
1  20/09/2018 12:03:15                   Cash            Food   
2           19/09/2018  Saving Bank account 1    subscription   
3  17/09/2018 23:41:17  Saving Bank account 1    subscription   
4  16/09/2018 17:15:08                   Cash       Festivals   

               Subcategory                         Note  Amount  \
0                    Train         2 Place 5 to Place 0    30.0   
1                   snacks  Idli medu Vada mix 2 plates    60.0   
2                  Netflix         1 month subscription   199.0   
3  Mobile Service Provider            Data booster pack    19.0   
4             Ganesh Pujan                  Ganesh idol   251.0   

  Income/Expense Currency  
0        Expense      INR  
1        Expense      INR  
2        Expense      INR  
3        Expense      INR  
4        Expense      INR  

Info Datas

Dataset kedua ini berisi catatan transaksi rumah tangga sehari-hari yang berasal dari India, sehingga nominal transaksinya dalam mata uang Rupee India (INR). Dataset ini memiliki kolom tambahan dibanding dataset pertama, yaitu subcategory, currency, income/expense, dan account.

## 2.2 Informasi Dataset

In [29]:
print("\nINFO DATASET")
print(df1.info())



INFO DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2461 entries, 0 to 2460
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            2461 non-null   object 
 1   Mode            2461 non-null   object 
 2   Category        2461 non-null   object 
 3   Subcategory     1826 non-null   object 
 4   Note            1940 non-null   object 
 5   Amount          2461 non-null   float64
 6   Income/Expense  2461 non-null   object 
 7   Currency        2461 non-null   object 
dtypes: float64(1), object(7)
memory usage: 153.9+ KB
None


Menampilkan ringkasan teknis dataset: jumlah baris, nama kolom, jumlah non-null, dan tipe data. Informasi ini menjadi acuan untuk menentukan kolom mana yang perlu dikonversi tipe datanya dan kolom mana yang memiliki missing value.

## 2.3 Missing Value

In [30]:
print("\nMissing Value:")
print(df1.isnull().sum())


Missing Value:
Date                0
Mode                0
Category            0
Subcategory       635
Note              521
Amount              0
Income/Expense      0
Currency            0
dtype: int64


Hasil tersebut menunjukkan bahwa sebagian besar kolom tidak memiliki data kosong karena nilainya 0. Namun, kolom Subcategory memiliki 635 data kosong dan kolom Note memiliki 521 data kosong. Pengecekan ini dilakukan untuk mengetahui apakah masih ada data yang belum lengkap sebelum proses pembersihan data dilakukan.

## 2.4 Duplicate

In [31]:
print("\nJumlah Duplicate:")
print(df1.duplicated().sum())


Jumlah Duplicate:
9


Hasil tersebut menunjukkan bahwa terdapat 9 data duplikat pada dataset. Data duplikat berarti ada beberapa baris data yang isinya sama persis dengan baris lainnya. Pengecekan ini dilakukan untuk mengetahui apakah terdapat data yang berulang sebelum proses pembersihan data dilakukan.


# 3. Cleaning

## 3.1 Hapus Duplicate


In [32]:
df1.drop_duplicates(inplace=True)

Duplikat langsung dihapus di awal proses cleaning agar tidak ada pemrosesan yang sia-sia pada data yang akan dibuang. inplace=True berarti perubahan langsung diterapkan pada df1 tanpa perlu menyimpan ke variabel baru.

## 3.2 Rapikan Nama Kolom

In [33]:
df1.columns = (
    df1.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

print("\nNama Kolom Setelah Dirapikan:")
print(df1.columns)


Nama Kolom Setelah Dirapikan:
Index(['date', 'mode', 'category', 'subcategory', 'note', 'amount',
       'income/expense', 'currency'],
      dtype='object')


Nama kolom pada datase Proses yang dilakukan yaitu menghapus spasi yang tidak diperlukan, mengubah huruf menjadi kecil semua, dan mengganti spasi dengan tanda underscore (_).

Hasilnya, nama kolom menjadi lebih rapi dan konsisten seperti date, mode, category, subcategory, dan lainnya sehingga lebih mudah digunakan saat proses analisis data.

## 3.3 Cleaning Text

In [34]:
for col in df1.select_dtypes(include='object').columns:
    df1[col] = (
        df1[col]
        .astype(str)
        .str.strip()
    )

Proses yang dilakukan yaitu mengubah seluruh data menjadi tipe string dan menghapus spasi berlebih di awal atau akhir teks menggunakan .str.strip().

Tujuannya agar data lebih rapi dan konsisten sehingga mengurangi kesalahan saat analisis atau pengolahan data selanjutnya.

## 3.4 Handle Missing Value

In [35]:
# ubah string kosong menjadi NaN
df1= df1.replace(r'^\s*$', np.nan, regex=True)

print("\nMissing Value Sebelum Cleaning:")
print(df1.isnull().sum())


Missing Value Sebelum Cleaning:
date              0
mode              0
category          0
subcategory       0
note              0
amount            0
income/expense    0
currency          0
dtype: int64


Mengubah string kosong menjadi nilai NaN agar dapat terdeteksi sebagai missing value. Setelah dilakukan pengecekan ulang menggunakan isnull().sum(), hasilnya menunjukkan bahwa semua kolom memiliki nilai 0, yang berarti tidak ada missing value pada dataset setelah proses cleaning dilakukan.

## 3.5 Handle Kolom Penting

In [36]:
# amount wajib ada
df1 = df1.dropna(subset=['amount'])

# date wajib ada
df1 = df1.dropna(subset=['date'])

# category kosong → other
df1['category'] = df1['category'].fillna('other')

# subcategory kosong → unknown
df1['subcategory'] = df1['subcategory'].fillna('unknown')

# note kosong → string kosong
df1['note'] = df1['note'].fillna('')

# currency kosong → INR
df1['currency'] = df1['currency'].fillna('INR')

Menangani data pada kolom-kolom penting agar dataset tetap lengkap dan konsisten. Data pada kolom amount dan date yang kosong akan dihapus karena kedua kolom tersebut wajib ada. Kemudian, nilai kosong pada kolom category diisi dengan other, kolom subcategory diisi dengan unknown, kolom note diisi string kosong (''), dan kolom currency diisi INR. Proses ini dilakukan agar tidak ada data penting yang kosong saat analisis dilakukan.

## 3.6 Filter hanya expense

In [37]:
df1 = df1[df1['income/expense'] == 'Expense']

Memfilter data agar hanya menampilkan transaksi dengan jenis Expense. Artinya, data pengeluaran saja yang akan disimpan di df1, sedangkan data selain pengeluaran seperti Income tidak akan ikut digunakan.

## 3.7 Merubah format tanggal

In [38]:
df1['tanggal_transaksi'] = pd.to_datetime(
    df1['date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
).dt.date

# hapus tanggal invalid
df1 = df1.dropna(subset=['tanggal_transaksi'])

Mengubah kolom date menjadi format tanggal yang benar menggunakan pd.to_datetime(). Parameter dayfirst=True digunakan agar format tanggal dibaca dengan urutan hari-bulan-tahun, sedangkan errors='coerce' akan mengubah tanggal yang tidak valid menjadi NaN. Setelah itu, data dengan tanggal tidak valid dihapus menggunakan dropna() agar dataset hanya berisi tanggal yang valid.

## 3.8 Konversi INR ke IDR

In [39]:
# kurs tetap
kurs_inr_to_idr = 190

# amount jadi numerik
df1['amount'] = pd.to_numeric(
    df1['amount'],
    errors='coerce'
)

# hapus amount invalid
df1 = df1.dropna(subset=['amount'])

# konversi ke rupiah
df1['total_pengeluaran'] = (
    df1['amount'] * kurs_inr_to_idr
).round(0).astype(int)

Mengonversi nilai mata uang dari INR ke IDR. Pertama, dibuat kurs tetap 1 INR = 190 IDR. Setelah itu, kolom amount diubah menjadi tipe numerik menggunakan pd.to_numeric(), lalu data yang memiliki nilai amount tidak valid dihapus. Selanjutnya, dilakukan perhitungan konversi ke rupiah dengan mengalikan amount dengan kurs INR ke IDR, kemudian hasilnya disimpan pada kolom total_pengeluaran dalam bentuk bilangan bulat.

## 3.9 Mapping kategori


In [40]:
# rapikan category
df1['category'] = (
    df1['category']
    .astype(str)
    .str.strip()
    .str.lower()
)

kategori_mapping = {

    # makanan
    'food': 'Makanan/Minuman',

    # kesehatan
    'health': 'Kesehatan',

    # hiburan
    'entertainment': 'Hiburan',
    'travel': 'Hiburan',

    # utilitas
    'subscription': 'Tagihan/Utilitas',

    # belanja
    'shopping': 'Belanja',

    # pendidikan
    'education': 'Pendidikan',

    # lainnya
    'transportation': 'Transportasi',
    'place': 'Transportasi',
    'household': 'Belanja',
    'other': 'Lain-lain'
}

df1['kategori'] = df1['category'].map(
    kategori_mapping
)

# kategori kosong → lain-lain
df1['kategori'] = df1['kategori'].fillna(
    'Lain-lain'
)

Merapikan dan mengelompokkan data pada kolom category. Isi kolom dibersihkan dengan menghapus spasi berlebih dan mengubah huruf menjadi kecil. Setelah itu, dilakukan pemetaan kategori menggunakan kategori_mapping, misalnya food menjadi Makanan/Minuman, health menjadi Kesehatan, dan entertainment menjadi Hiburan. Beberapa kategori lain seperti transportation, shopping, dan travel digabung ke kategori Lain-lain. Jika masih ada kategori kosong setelah proses mapping, nilainya akan diisi menjadi Lain-lain.

## 3.10 Membuat kolom nama produk

In [41]:
df1['nama_produk'] = np.where(
    (df1['note'].isna()) |
    (df1['note'] == '') |
    (df1['note'] == 'nan'),

    df1['subcategory'],
    df1['note']
)

Membuat kolom baru bernama nama_produk. Jika kolom note kosong, bernilai string kosong (''), atau berisi 'nan', maka nilai nama_produk akan diambil dari kolom subcategory. Namun, jika note memiliki isi, maka isi dari note tersebut yang akan digunakan sebagai nama_produk. Tujuannya agar setiap data memiliki nama produk yang lebih lengkap dan mudah dikenali.

## 3.11 Cleaning nama produk

In [42]:
df1['nama_produk'] = (
    df1['nama_produk']
    .astype(str)
    .str.lower()
    .str.strip()
)

Membersihkan data pada kolom nama_produk. Proses yang dilakukan yaitu mengubah semua teks menjadi huruf kecil menggunakan .str.lower() dan menghapus spasi berlebih di awal atau akhir teks dengan .str.strip(). Tujuannya agar penulisan nama produk menjadi lebih rapi dan konsisten.

## 3.13 Pilih kolom final

In [43]:
df1_final = df1[[
    'tanggal_transaksi',
    'nama_produk',
    'kategori',
    'total_pengeluaran'
]]


Memilih kolom-kolom yang akan digunakan sebagai dataset akhir. Kolom yang dipilih yaitu tanggal_transaksi, nama_produk, kategori, dan total_pengeluaran lalu disimpan ke dalam DataFrame baru bernama df1_final. Tujuannya agar dataset akhir menjadi lebih rapi dan hanya berisi informasi yang diperlukan untuk analisis.

## 3.14 Hapus missing value data final

In [44]:
# ubah string kosong jadi NaN
df1_final['nama_produk'] = df1_final['nama_produk'].replace(
    r'^\s*$',
    np.nan,
    regex=True
)

# hapus missing value
df1_final = df1_final.dropna(
    subset=[
        'tanggal_transaksi',
        'nama_produk',
        'kategori',
        'total_pengeluaran'
    ]
)



/tmp/ipykernel_5075/1324916229.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1_final['nama_produk'] = df1_final['nama_produk'].replace(


Membersihkan missing value pada dataset akhir df1_final. Pertama, string kosong pada kolom nama_produk diubah menjadi NaN agar dapat terdeteksi sebagai missing value. Setelah itu, data yang masih memiliki nilai kosong pada kolom penting seperti tanggal_transaksi, nama_produk, kategori, dan total_pengeluaran akan dihapus menggunakan dropna(). Tujuannya agar dataset final bersih dan siap digunakan untuk analisis.

## Hapus duplicate dataset final

In [45]:
df1_final = df1_final.drop_duplicates()

Menghapus data duplikat pada dataset akhir df1_final menggunakan drop_duplicates(). Proses ini dilakukan agar tidak ada baris data yang sama atau berulang, sehingga dataset menjadi lebih bersih dan hasil analisis lebih akurat.

## Hasil dataset 2 bersih

In [46]:
print("\n" + "=" * 60)
print("HASIL CLEANING")
print("=" * 60)

print("\nUkuran Dataset 2:")
print(df1_final.shape)

print("\n5 Data Teratas Dataset 2:")
print(df1_final.head())

print("\nInfo Dataset Final 2:")
print(df1_final.info())

print("\nMissing Value:")
print(df1_final.isnull().sum())

print("\nJumlah Duplicate:")
print(df1_final.duplicated().sum())


HASIL CLEANING

Ukuran Dataset 2:
(2168, 4)

5 Data Teratas Dataset 2:
  tanggal_transaksi                  nama_produk          kategori  \
0        2018-09-20         2 place 5 to place 0      Transportasi   
1        2018-09-20  idli medu vada mix 2 plates   Makanan/Minuman   
2        2018-09-19         1 month subscription  Tagihan/Utilitas   
3        2018-09-17            data booster pack  Tagihan/Utilitas   
4        2018-09-16                  ganesh idol         Lain-lain   

   total_pengeluaran  
0               5700  
1              11400  
2              37810  
3               3610  
4              47690  

Info Dataset Final 2:
<class 'pandas.core.frame.DataFrame'>
Index: 2168 entries, 0 to 2460
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   tanggal_transaksi  2168 non-null   object
 1   nama_produk        2168 non-null   object
 2   kategori           2168 non-null   object
 3   total

Hasil tersebut menunjukkan bahwa proses cleaning pada dataset akhir berhasil dilakukan. Dataset final memiliki ukuran (2168, 4) yang berarti terdapat 2168 baris data dan 4 kolom. Lima data teratas ditampilkan sebagai contoh isi dataset setelah dibersihkan dan dirapikan.

Pada bagian informasi dataset (info()), terlihat bahwa semua kolom memiliki 2168 non-null, yang berarti sudah tidak ada missing value pada dataset akhir. Tipe data juga sudah sesuai, yaitu kolom tanggal_transaksi, nama_produk, dan kategori bertipe object, sedangkan total_pengeluaranbertipe int64. Hal ini menunjukkan dataset sudah bersih, konsisten, dan siap digunakan untuk analisis lebih lanjut.

# Simpan Dataset 2

In [47]:
output_file = 'clean_daily_household_transactions.csv'

df1_final.to_csv(
    output_file,
    index=False
)

print("\nFile berhasil disimpan:")
print(output_file)


File berhasil disimpan:
clean_daily_household_transactions.csv


Dataset 2 telah disimpan dalam format csv dengan nama clean_daily_household_transactions.csv



# Validasi akhir untuk dataset 2

In [48]:
print("\n" + "=" * 60)
print("VALIDASI AKHIR")
print("=" * 60)

print("\nContoh Data Final:")
print(df1_final.head())

print("\nKategori:")
print(df1_final['kategori'].value_counts())

print("\nRentang Nominal:")
print(df1_final['total_pengeluaran'].describe())


VALIDASI AKHIR

Contoh Data Final:
  tanggal_transaksi                  nama_produk          kategori  \
0        2018-09-20         2 place 5 to place 0      Transportasi   
1        2018-09-20  idli medu vada mix 2 plates   Makanan/Minuman   
2        2018-09-19         1 month subscription  Tagihan/Utilitas   
3        2018-09-17            data booster pack  Tagihan/Utilitas   
4        2018-09-16                  ganesh idol         Lain-lain   

   total_pengeluaran  
0               5700  
1              11400  
2              37810  
3               3610  
4              47690  

Kategori:
kategori
Makanan/Minuman     905
Lain-lain           527
Transportasi        306
Belanja             176
Tagihan/Utilitas    142
Kesehatan            94
Pendidikan           18
Name: count, dtype: int64

Rentang Nominal:
count    2.168000e+03
mean     1.695842e+05
std      7.286395e+05
min      3.800000e+02
25%      5.700000e+03
50%      1.567500e+04
75%      7.006250e+04
max      1.900000e+

# Menggabungkan kedua dataset

In [49]:
# load kedua dataset
dataset1 = pd.read_csv('/content/cleaned_budgetwise_finance_dataset.csv')
dataset2 = pd.read_csv('/content/clean_daily_household_transactions.csv')

# gabungkan
final_dataset = pd.concat(
    [dataset1, dataset2],
    ignore_index=True
)

# 1. konversi ke numerik
final_dataset['total_pengeluaran'] = pd.to_numeric(
    final_dataset['total_pengeluaran'], errors='coerce'
)

# 2. filter hapus nilai 0 dan NaN
final_dataset = final_dataset[final_dataset['total_pengeluaran'] > 0]

# cek hasil
print(final_dataset.shape)

(9264, 4)


## Menghapus Missing Value setelah data digabungkan

In [50]:
final_dataset = final_dataset.dropna(subset=[
    'tanggal_transaksi',
    'nama_produk',
    'kategori',
    'total_pengeluaran'
])

## Hapus duplikat dataset yang telah digabungkan

In [51]:
final_dataset = final_dataset.drop_duplicates().reset_index(drop=True)


In [52]:
# Validasi
print("Shape:", final_dataset.shape)
print("Missing Value:\n", final_dataset.isnull().sum())
print("Nilai 0:", (final_dataset['total_pengeluaran'] == 0).sum())

Shape: (9183, 4)
Missing Value:
 tanggal_transaksi    0
nama_produk          0
kategori             0
total_pengeluaran    0
dtype: int64
Nilai 0: 0


## Menyimpan dataset yang sudah digabungkan

In [53]:
# simpan
final_dataset.to_csv('final_finance_dataset.csv', index=False)
print("Dataset berhasil disimpan!")

print("Dataset berhasil digabung!")

Dataset berhasil disimpan!
Dataset berhasil digabung!


Menunjukkan bahwa dataset akhir berhasil digabung dengan ukuran 9302 baris dan 4 kolom. Pesan "Dataset berhasil digabung!" menandakan proses penggabungan dan penyimpanan dataset berjalan dengan sukses. Dataset akhir kemudian disimpan dengan nama final_finance_dataset.csv.